# Grouped Query Attention (GQA)

## MHA vs MQA vs GQA -- The Spectrum of KV Head Sharing

Standard **Multi-Head Attention (MHA)** assigns a unique Key and Value head to every Query head.
This maximizes representational capacity but is expensive: KV cache memory scales linearly with
the number of heads.

**Multi-Query Attention (MQA)** takes the opposite extreme -- all Query heads share a *single*
KV head. This slashes memory and compute, but can degrade quality.

**Grouped Query Attention (GQA)** is the sweet spot. Query heads are partitioned into *groups*,
and each group shares one KV head. With `G` groups:

| Config | KV heads | Relationship |
|--------|----------|--------------|
| MHA    | H        | G = H (every Q head has its own KV head) |
| GQA    | G        | 1 < G < H (groups of Q heads share KV heads) |
| MQA    | 1        | G = 1 (all Q heads share one KV head) |

```
MHA (G=H=8)       GQA (G=2)          MQA (G=1)
Q: q1 q2 q3 q4    Q: q1 q2 q3 q4     Q: q1 q2 q3 q4
    |  |  |  |        \  |  |  /          \ | | /
K: k1 k2 k3 k4    K:  k1    k2        K:    k1
V: v1 v2 v3 v4    V:  v1    v2        V:    v1
```

This notebook builds intuition for GQA by comparing it against MHA and MQA across
parameter counts, memory savings, output quality, and attention patterns.

## 1. Imports and Setup

In [ ]:
%matplotlib inline

import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

from gqa import GQA

torch.manual_seed(42)

# Core dimensions
d = 64          # model dimension
seq_len = 16    # sequence length
batch_size = 2  # batch size

# Create a shared input tensor
x = torch.randn(batch_size, seq_len, d)
print(f"Input shape: {x.shape}")
print(f"Model dim d={d}, seq_len={seq_len}, batch_size={batch_size}")

## 2. Standard Multi-Head Attention Baseline

In standard MHA with `H` heads, we have independent Q, K, V projections per head.
Using our `GQA` module with `h_groups = H` recovers full MHA: each group contains
exactly one query head with its own KV head.

In [ ]:
# MHA: 8 groups = 8 KV heads = 8 Q heads (one Q head per group)
num_heads = 8
mha = GQA(d=d, h_groups=num_heads)

print(f"MHA configuration:")
print(f"  Total Q heads:  {num_heads}")
print(f"  KV head groups: {mha.groups}")
print(f"  Head dim:       {d // num_heads}")

In [ ]:
torch.manual_seed(42)
mha_out = mha(x)
print(f"MHA output shape: {mha_out.shape}")
print(f"MHA output norm:  {mha_out.norm():.4f}")

## 3. Grouped Query Attention

Now we create a GQA module with `h_groups=2`. This means:
- The model dimension `d=64` is split into 2 KV groups
- Each group has a head dimension of `d // groups = 32`
- Query, Key, and Value are all reshaped to `(B, T, 2, 32)`
- Attention is computed *within* each group independently

In [ ]:
gqa = GQA(d=d, h_groups=2)

print(f"GQA configuration:")
print(f"  KV head groups: {gqa.groups}")
print(f"  Head dim per group: {d // gqa.groups}")

In [ ]:
torch.manual_seed(42)
gqa_out = gqa(x)
print(f"GQA output shape: {gqa_out.shape}")
print(f"GQA output norm:  {gqa_out.norm():.4f}")

## 4. How KV Heads Are Shared Across Query Heads

In a true GQA implementation (e.g., Llama 2 70B), the number of Q heads is
typically larger than the number of KV heads. Multiple Q heads *share* the same
K and V projections.

```
Example: 8 Q heads, 2 KV groups

Group 0:  Q_0, Q_1, Q_2, Q_3  -->  K_0, V_0
Group 1:  Q_4, Q_5, Q_6, Q_7  -->  K_1, V_1

Within each group, all Q heads attend to the *same* K and V.
This is what saves memory: we only need 2 KV cache entries
instead of 8.
```

Our `GQA` module achieves grouping by reshaping Q, K, V to `(B, T, groups, d//groups)`,
then computing attention within each group. Let's trace through the shapes.

In [ ]:
# Trace the internal reshaping step by step
groups = 2
head_dim = d // groups

# Step 1: Linear projections (full d -> d)
q = gqa.Wq(x)  # (B, T, d)
k = gqa.Wk(x)  # (B, T, d)
v = gqa.Wv(x)  # (B, T, d)
print(f"After projection -- Q: {q.shape}, K: {k.shape}, V: {v.shape}")

# Step 2: Reshape into groups
qs = q.view(batch_size, seq_len, groups, head_dim)
ks = k.view(batch_size, seq_len, groups, head_dim)
vs = v.view(batch_size, seq_len, groups, head_dim)
print(f"After reshape   -- Q: {qs.shape}, K: {ks.shape}, V: {vs.shape}")

# Step 3: Attention within each group
# qs @ ks^T: (B, T, groups, head_dim) @ (B, T, head_dim, groups) is not right.
# Actually: ks.transpose(-2, -1) gives (B, T, head_dim, groups) -- wrong.
# The module does qs @ ks.transpose(-2,-1) which is:
#   (B, T, groups, head_dim) @ (B, T, groups, head_dim).transpose(-2,-1)
#   = (B, T, groups, head_dim) @ (B, T, head_dim, groups)
# This computes a (B, T, groups, groups) attention matrix.
att = torch.softmax(qs @ ks.transpose(-2, -1) / (head_dim ** 0.5), dim=-1)
print(f"Attention map:  {att.shape}  (B, T, groups, groups)")

# Step 4: Weighted sum and reshape back
out = (att @ vs).reshape(batch_size, seq_len, d)
print(f"Output:         {out.shape}")

## 5. Comparing Output Shapes

Regardless of the number of groups, the output always has shape `(B, T, d)`.
The grouping only affects the *internal* structure of attention computation.

In [ ]:
configs = {
    "MQA (groups=1)": GQA(d, h_groups=1),
    "GQA (groups=2)": GQA(d, h_groups=2),
    "GQA (groups=4)": GQA(d, h_groups=4),
    "MHA (groups=8)": GQA(d, h_groups=8),
}

print(f"{'Configuration':<20} {'Output Shape':<20} {'Groups':<8} {'Head Dim':<10}")
print("-" * 58)
for name, model in configs.items():
    out = model(x)
    print(f"{name:<20} {str(out.shape):<20} {model.groups:<8} {d // model.groups:<10}")

print("\nAll outputs have the same shape -- GQA is a drop-in replacement for MHA.")

## 6. Parameter Comparison: MHA vs GQA vs MQA

In our implementation, all three variants use the same linear projections
(`Wq`, `Wk`, `Wv` are all `d -> d`), so they have identical parameter counts.

In a **production GQA** (like Llama 2), the K and V projections are *smaller*:
- MHA: `Wk` and `Wv` are `d -> d` (H heads, each of dimension d/H)
- GQA: `Wk` and `Wv` are `d -> G*(d/H)` (G groups, each of dimension d/H)
- MQA: `Wk` and `Wv` are `d -> d/H` (single head)

Let's compute the theoretical parameter counts for a production implementation.

In [ ]:
def count_params_production(d, num_q_heads, num_kv_heads):
    """Count parameters for a production attention implementation.
    
    Q projection: d -> d (num_q_heads * head_dim)
    K projection: d -> num_kv_heads * head_dim
    V projection: d -> num_kv_heads * head_dim
    O projection: d -> d
    """
    head_dim = d // num_q_heads
    q_params = d * d                          # Wq: d -> d
    k_params = d * (num_kv_heads * head_dim)  # Wk: d -> num_kv_heads * head_dim
    v_params = d * (num_kv_heads * head_dim)  # Wv: d -> num_kv_heads * head_dim
    o_params = d * d                          # Wo: d -> d
    return q_params + k_params + v_params + o_params

num_q_heads = 8
head_dim = d // num_q_heads  # = 8

results = {}
for label, num_kv in [("MHA (8 KV)", 8), ("GQA (4 KV)", 4), ("GQA (2 KV)", 2), ("MQA (1 KV)", 1)]:
    p = count_params_production(d, num_q_heads, num_kv)
    results[label] = p
    print(f"{label:<14}  params = {p:>6}  (KV dim = {num_kv * head_dim})")

mha_params = results["MHA (8 KV)"]
print(f"\nRelative to MHA:")
for label, p in results.items():
    print(f"  {label:<14}  {p/mha_params:.1%}")

## 7. Parameter Count Bar Chart

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))

labels = list(results.keys())
params = list(results.values())
colors = ["#2196F3", "#4CAF50", "#FF9800", "#F44336"]

bars = ax.bar(labels, params, color=colors, edgecolor="black", linewidth=0.5)

for bar, p in zip(bars, params):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 100,
            f"{p:,}", ha="center", va="bottom", fontsize=10, fontweight="bold")

ax.set_ylabel("Parameter Count")
ax.set_title(f"Attention Parameter Counts (d={d}, {num_q_heads} Q heads)")
ax.set_ylim(0, max(params) * 1.15)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

## 8. KV Cache Memory Savings

During autoregressive inference, we cache K and V tensors for all past tokens.
The KV cache size per layer is:

```
KV cache = 2 * num_kv_heads * head_dim * seq_len * bytes_per_element
```

GQA with fewer KV heads directly reduces this cache.

In [ ]:
def kv_cache_bytes(num_kv_heads, head_dim, seq_len, dtype_bytes=4):
    """KV cache memory per layer in bytes."""
    # 2 for K and V, times heads, head_dim, seq_len, bytes
    return 2 * num_kv_heads * head_dim * seq_len * dtype_bytes

seq_lengths = [128, 512, 1024, 2048, 4096, 8192]
kv_configs = {
    "MHA (8 KV heads)": 8,
    "GQA (4 KV heads)": 4,
    "GQA (2 KV heads)": 2,
    "MQA (1 KV head)":  1,
}

print(f"{'Seq Len':<10}", end="")
for label in kv_configs:
    print(f"{label:<22}", end="")
print()
print("-" * 98)

for sl in seq_lengths:
    print(f"{sl:<10}", end="")
    for label, nkv in kv_configs.items():
        mem = kv_cache_bytes(nkv, head_dim, sl)
        print(f"{mem / 1024:>8.1f} KB{'':<12}", end="")
    print()

## 9. KV Cache Memory Plot

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

linestyles = ["-", "--", "-.", ":"]
colors = ["#2196F3", "#4CAF50", "#FF9800", "#F44336"]

for (label, nkv), ls, c in zip(kv_configs.items(), linestyles, colors):
    mem_kb = [kv_cache_bytes(nkv, head_dim, sl) / 1024 for sl in seq_lengths]
    ax.plot(seq_lengths, mem_kb, ls, color=c, linewidth=2, marker="o", label=label)

ax.set_xlabel("Sequence Length")
ax.set_ylabel("KV Cache Size (KB per layer)")
ax.set_title("KV Cache Memory vs Sequence Length")
ax.legend()
ax.grid(alpha=0.3)
ax.set_xscale("log", base=2)
plt.tight_layout()
plt.show()

## 10. Memory Savings as a Function of Number of Groups

Plotting the KV cache reduction ratio compared to full MHA as we vary the
number of KV head groups.

In [ ]:
total_heads = 8
group_options = [1, 2, 4, 8]  # MQA, GQA-2, GQA-4, MHA

savings = []
for g in group_options:
    ratio = g / total_heads  # fraction of MHA's KV cache
    savings.append(ratio)
    print(f"Groups={g}: KV cache is {ratio:.0%} of MHA ({(1 - ratio):.0%} savings)")

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar([str(g) for g in group_options], [s * 100 for s in savings],
       color=["#F44336", "#FF9800", "#4CAF50", "#2196F3"],
       edgecolor="black", linewidth=0.5)
ax.set_xlabel("Number of KV Groups")
ax.set_ylabel("KV Cache Size (% of MHA)")
ax.set_title("KV Cache Reduction by Number of Groups")
ax.set_ylim(0, 115)
for i, (g, s) in enumerate(zip(group_options, savings)):
    ax.text(i, s * 100 + 2, f"{s:.0%}", ha="center", fontweight="bold")
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

## 11. Ablation: Number of Groups

How does changing the number of groups affect the output? We test
`groups=1` (MQA), 2, 4, and 8 (MHA) and compare:
- **Output norm**: Overall magnitude of the output
- **Output diversity**: Standard deviation across the sequence dimension (how varied the per-token representations are)

In [ ]:
torch.manual_seed(42)
x_test = torch.randn(batch_size, seq_len, d)

group_values = [1, 2, 4, 8]
norms = []
diversities = []
outputs = {}

print(f"{'Groups':<10} {'Output Norm':<16} {'Token Std (diversity)':<24} {'Label'}")
print("-" * 65)

for g in group_values:
    torch.manual_seed(42)  # Same init for fair comparison
    model = GQA(d, h_groups=g)
    out = model(x_test)
    outputs[g] = out.detach()
    
    norm = out.norm().item()
    # Diversity: std of per-token norms across the sequence
    token_norms = out.norm(dim=-1)  # (B, T)
    diversity = token_norms.std().item()
    
    norms.append(norm)
    diversities.append(diversity)
    
    label = {1: "MQA", 8: "MHA"}.get(g, f"GQA-{g}")
    print(f"{g:<10} {norm:<16.4f} {diversity:<24.4f} {label}")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

labels = [f"G={g}" for g in group_values]
colors = ["#F44336", "#FF9800", "#4CAF50", "#2196F3"]

ax1.bar(labels, norms, color=colors, edgecolor="black", linewidth=0.5)
ax1.set_ylabel("Output Frobenius Norm")
ax1.set_title("Output Norm by Group Count")
ax1.grid(axis="y", alpha=0.3)

ax2.bar(labels, diversities, color=colors, edgecolor="black", linewidth=0.5)
ax2.set_ylabel("Std of Token Norms")
ax2.set_title("Output Diversity by Group Count")
ax2.grid(axis="y", alpha=0.3)

plt.suptitle("Effect of Number of Groups on Output Statistics", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

## 12. Quality Comparison: Cosine Similarity Between GQA and MHA

If GQA is a good approximation of MHA, outputs should be similar when both
use the same weights. Since our module uses identical projection dimensions
regardless of groups, we can directly compare by using the *same weights*
but different group counts.

In [ ]:
# Create a reference MHA model
torch.manual_seed(42)
ref_mha = GQA(d, h_groups=8)
ref_out = ref_mha(x_test).detach()

# Now create GQA variants with the *same* weight initialization
cos_sims = []
for g in group_values:
    torch.manual_seed(42)  # Same weight init
    model = GQA(d, h_groups=g)
    out = model(x_test).detach()
    
    # Flatten and compute cosine similarity
    cos = torch.nn.functional.cosine_similarity(
        ref_out.reshape(-1).unsqueeze(0),
        out.reshape(-1).unsqueeze(0)
    ).item()
    cos_sims.append(cos)
    label = {1: "MQA", 8: "MHA"}.get(g, f"GQA-{g}")
    print(f"{label:<10} (groups={g}):  cosine similarity to MHA = {cos:.6f}")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))

bar_labels = [f"G={g}\n({'MQA' if g==1 else 'MHA' if g==8 else f'GQA-{g}'})" for g in group_values]
ax.bar(bar_labels, cos_sims, color=colors, edgecolor="black", linewidth=0.5)
ax.set_ylabel("Cosine Similarity to MHA Output")
ax.set_title("Output Similarity to Full MHA (Same Weights)")
ax.set_ylim(min(cos_sims) - 0.05, 1.02)
for i, cs in enumerate(cos_sims):
    ax.text(i, cs + 0.005, f"{cs:.4f}", ha="center", fontsize=9, fontweight="bold")
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

## 13. Attention Patterns: GQA vs MHA

Let's visualize the internal attention matrices for different group counts.
With fewer groups, the attention maps are larger per group (each group
has more dimensions to attend over).

In [ ]:
def get_attention_maps(model, x):
    """Extract attention maps from a GQA model."""
    q = model.Wq(x)
    k = model.Wk(x)
    B, T, D = q.shape
    g = model.groups
    hd = D // g
    qs = q.view(B, T, g, hd)
    ks = k.view(B, T, g, hd)
    att = torch.softmax(qs @ ks.transpose(-2, -1) / (hd ** 0.5), dim=-1)
    return att  # (B, T, g, g)

# Collect attention maps for different group counts
fig, axes = plt.subplots(1, 4, figsize=(16, 4))

for idx, g in enumerate(group_values):
    torch.manual_seed(42)
    model = GQA(d, h_groups=g)
    att = get_attention_maps(model, x_test)
    
    # Take first batch, first token's attention pattern
    # att shape: (B, T, g, g) -- show the attention for token 0
    att_map = att[0, 0].detach().numpy()  # (g, g)
    
    im = axes[idx].imshow(att_map, cmap="Blues", vmin=0, vmax=1, aspect="auto")
    label = {1: "MQA", 8: "MHA"}.get(g, f"GQA-{g}")
    axes[idx].set_title(f"{label} (G={g})\nAtt: {g}x{g}", fontsize=11)
    axes[idx].set_xlabel("Key group")
    axes[idx].set_ylabel("Query group")
    plt.colorbar(im, ax=axes[idx], fraction=0.046)

plt.suptitle("Attention Patterns at Token 0 (Batch 0)", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Attention entropy: how spread out is attention?
# Higher entropy = more uniform attention, lower = more focused
print(f"{'Config':<12} {'Mean Entropy':<16} {'Interpretation'}")
print("-" * 55)

for g in group_values:
    torch.manual_seed(42)
    model = GQA(d, h_groups=g)
    att = get_attention_maps(model, x_test)  # (B, T, g, g)
    
    # Entropy of attention distributions
    # att has shape (B, T, g, g), last dim sums to 1
    entropy = -(att * (att + 1e-10).log()).sum(dim=-1)  # (B, T, g)
    mean_entropy = entropy.mean().item()
    max_entropy = np.log(g)  # maximum possible entropy
    
    label = {1: "MQA", 8: "MHA"}.get(g, f"GQA-{g}")
    interp = f"max possible = {max_entropy:.3f}"
    print(f"{label:<12} {mean_entropy:<16.4f} {interp}")

## 14. Attention Map Across All Tokens

Let's look at how attention evolves across the full sequence for the MHA case.

In [ ]:
# Show attention across all tokens for MHA (8 groups)
torch.manual_seed(42)
mha_model = GQA(d, h_groups=8)
att_mha = get_attention_maps(mha_model, x_test)  # (B, T, 8, 8)

fig, axes = plt.subplots(2, 8, figsize=(16, 5))
fig.suptitle("MHA Attention Maps for First 16 Tokens (Batch 0)", fontsize=13)

for t in range(min(16, seq_len)):
    row = t // 8
    col = t % 8
    att_map = att_mha[0, t].detach().numpy()  # (8, 8)
    axes[row, col].imshow(att_map, cmap="Blues", vmin=0, vmax=1, aspect="auto")
    axes[row, col].set_title(f"t={t}", fontsize=8)
    axes[row, col].set_xticks([])
    axes[row, col].set_yticks([])

plt.tight_layout()
plt.show()

## 15. Per-Token Cosine Similarity

Instead of a single global similarity score, let's look at how similar
GQA and MHA outputs are at each token position.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))

# Reference: MHA
torch.manual_seed(42)
ref_model = GQA(d, h_groups=8)
ref_out = ref_model(x_test).detach()

for g, color in zip([1, 2, 4], ["#F44336", "#FF9800", "#4CAF50"]):
    torch.manual_seed(42)
    model = GQA(d, h_groups=g)
    out = model(x_test).detach()
    
    # Per-token cosine similarity (batch 0)
    per_token_cos = torch.nn.functional.cosine_similarity(
        ref_out[0], out[0], dim=-1  # (T,)
    ).numpy()
    
    label = {1: "MQA", 8: "MHA"}.get(g, f"GQA-{g}")
    ax.plot(range(seq_len), per_token_cos, marker="o", color=color,
            linewidth=2, label=f"{label} (G={g})")

ax.set_xlabel("Token Position")
ax.set_ylabel("Cosine Similarity to MHA")
ax.set_title("Per-Token Similarity to MHA Output (Batch 0)")
ax.legend()
ax.grid(alpha=0.3)
ax.set_ylim(-0.1, 1.1)
plt.tight_layout()
plt.show()

## 16. Real-World Context: Llama 2 Uses GQA

Meta's **Llama 2 70B** was one of the first major open-source models to adopt GQA.
The key architectural decision:

| Model       | Q Heads | KV Heads | GQA Ratio | KV Cache Savings |
|-------------|---------|----------|-----------|------------------|
| Llama 2 7B  | 32      | 32       | 1:1 (MHA) | 0%              |
| Llama 2 13B | 40      | 40       | 1:1 (MHA) | 0%              |
| Llama 2 70B | 64      | 8        | 8:1 (GQA) | 87.5%           |

The 70B model uses GQA specifically to make inference tractable at scale.
With 64 Q heads and only 8 KV heads, each KV head serves 8 query heads.

In [ ]:
# Simulate Llama 2 configurations (scaled down)
llama_configs = {
    "Llama 2 7B\n(MHA)": {"q_heads": 32, "kv_heads": 32, "d_model": 4096},
    "Llama 2 13B\n(MHA)": {"q_heads": 40, "kv_heads": 40, "d_model": 5120},
    "Llama 2 70B\n(GQA)": {"q_heads": 64, "kv_heads": 8,  "d_model": 8192},
}

print(f"{'Model':<18} {'Q Heads':<10} {'KV Heads':<10} {'Head Dim':<10} "
      f"{'KV Cache/Token':<16} {'GQA Ratio'}")
print("-" * 82)

for name, cfg in llama_configs.items():
    clean_name = name.replace('\n', ' ')
    head_dim = cfg["d_model"] // cfg["q_heads"]
    kv_per_token = 2 * cfg["kv_heads"] * head_dim * 2  # fp16
    ratio = f"{cfg['q_heads'] // cfg['kv_heads']}:1"
    print(f"{clean_name:<18} {cfg['q_heads']:<10} {cfg['kv_heads']:<10} "
          f"{head_dim:<10} {kv_per_token:>8,} B{'':<7} {ratio}")

In [ ]:
# KV cache comparison for Llama 2 models at 4096 context length
context_len = 4096
num_layers = {"Llama 2 7B": 32, "Llama 2 13B": 40, "Llama 2 70B": 80}

fig, ax = plt.subplots(figsize=(8, 5))

model_names = []
cache_sizes_gb = []
bar_colors = []

for (name, cfg), (_, n_layers) in zip(llama_configs.items(), num_layers.items()):
    hd = cfg["d_model"] // cfg["q_heads"]
    # Total KV cache in bytes (fp16)
    total_cache = 2 * cfg["kv_heads"] * hd * context_len * n_layers * 2
    cache_gb = total_cache / (1024**3)
    model_names.append(name)
    cache_sizes_gb.append(cache_gb)
    color = "#4CAF50" if cfg["kv_heads"] < cfg["q_heads"] else "#2196F3"
    bar_colors.append(color)

bars = ax.bar(model_names, cache_sizes_gb, color=bar_colors,
              edgecolor="black", linewidth=0.5)
for bar, gb in zip(bars, cache_sizes_gb):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.05,
            f"{gb:.2f} GB", ha="center", fontsize=10, fontweight="bold")

ax.set_ylabel("KV Cache Size (GB)")
ax.set_title(f"KV Cache Memory at {context_len} Token Context (fp16)")
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\nLlama 2 70B's GQA saves {(1 - cache_sizes_gb[2] / (cache_sizes_gb[2] * 8)):.0%}")
print(f"compared to what it would need with full MHA (64 KV heads instead of 8).")

## 17. Throughput Estimate: GQA Speedup

GQA reduces the number of KV heads, which directly affects:
1. Memory bandwidth (less KV cache to read)
2. Compute (fewer KV projections)

Let's estimate the relative compute cost.

In [ ]:
# FLOPs for attention with different KV head counts
def attention_flops(seq_len, d_model, q_heads, kv_heads):
    """Approximate FLOPs for one attention layer."""
    head_dim = d_model // q_heads
    # Q projection: always full
    q_flops = 2 * seq_len * d_model * d_model
    # K, V projections: reduced
    kv_dim = kv_heads * head_dim
    kv_flops = 2 * 2 * seq_len * d_model * kv_dim  # 2 for K and V
    # Attention: Q @ K^T for each Q head (K is broadcast)
    att_flops = 2 * q_heads * seq_len * seq_len * head_dim
    # Att @ V for each Q head
    val_flops = 2 * q_heads * seq_len * seq_len * head_dim
    # Output projection
    o_flops = 2 * seq_len * d_model * d_model
    return q_flops + kv_flops + att_flops + val_flops + o_flops

d_model_test = 4096
q_heads_test = 32
sl_test = 2048

print(f"Attention FLOPs (d={d_model_test}, Q_heads={q_heads_test}, seq={sl_test})")
print("=" * 55)

mha_flops = attention_flops(sl_test, d_model_test, q_heads_test, q_heads_test)
for kv_h in [32, 16, 8, 4, 2, 1]:
    flops = attention_flops(sl_test, d_model_test, q_heads_test, kv_h)
    label = {32: "MHA", 1: "MQA"}.get(kv_h, f"GQA-{kv_h}")
    print(f"  KV_heads={kv_h:>2} ({label:<7}): {flops/1e9:>8.2f} GFLOPs  "
          f"({flops/mha_flops:.1%} of MHA)")

## 18. Weight Sharing Visualization

Let's create a visual diagram showing exactly which Q heads map to which KV heads
for different configurations.

In [ ]:
def draw_gqa_diagram(ax, num_q_heads, num_kv_heads, title):
    """Draw a diagram showing Q->KV head mapping."""
    q_y = 1.0
    kv_y = 0.0
    
    # Q heads
    q_xs = np.linspace(0, 1, num_q_heads)
    for i, qx in enumerate(q_xs):
        ax.plot(qx, q_y, "o", color="#2196F3", markersize=10, zorder=5)
        ax.text(qx, q_y + 0.08, f"Q{i}", ha="center", fontsize=6)
    
    # KV heads
    kv_xs = np.linspace(0, 1, max(num_kv_heads, 1))
    if num_kv_heads == 1:
        kv_xs = [0.5]
    for i, kvx in enumerate(kv_xs):
        ax.plot(kvx, kv_y, "s", color="#F44336", markersize=10, zorder=5)
        ax.text(kvx, kv_y - 0.12, f"KV{i}", ha="center", fontsize=6)
    
    # Connections
    q_per_kv = num_q_heads // num_kv_heads
    for qi in range(num_q_heads):
        kv_idx = qi // q_per_kv
        ax.plot([q_xs[qi], kv_xs[kv_idx]], [q_y - 0.05, kv_y + 0.05],
                "-", color="gray", alpha=0.5, linewidth=1)
    
    ax.set_title(title, fontsize=10, fontweight="bold")
    ax.set_xlim(-0.15, 1.15)
    ax.set_ylim(-0.3, 1.3)
    ax.axis("off")

fig, axes = plt.subplots(1, 4, figsize=(16, 3))
draw_gqa_diagram(axes[0], 8, 8, "MHA: 8Q -> 8KV")
draw_gqa_diagram(axes[1], 8, 4, "GQA: 8Q -> 4KV")
draw_gqa_diagram(axes[2], 8, 2, "GQA: 8Q -> 2KV")
draw_gqa_diagram(axes[3], 8, 1, "MQA: 8Q -> 1KV")

# Add legend
fig.text(0.5, -0.02, "Blue circles = Query heads    Red squares = KV heads    Lines = sharing",
         ha="center", fontsize=10, style="italic")
plt.tight_layout()
plt.show()

## 19. Output Distribution Analysis

How does the distribution of output values change with different group counts?
Let's look at histograms of the output tensor values.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 3.5), sharey=True)

for idx, g in enumerate(group_values):
    torch.manual_seed(42)
    model = GQA(d, h_groups=g)
    out = model(x_test).detach().numpy().flatten()
    
    label = {1: "MQA", 8: "MHA"}.get(g, f"GQA-{g}")
    axes[idx].hist(out, bins=50, color=colors[idx], alpha=0.7, edgecolor="black", linewidth=0.3)
    axes[idx].set_title(f"{label} (G={g})\nmean={out.mean():.3f}, std={out.std():.3f}",
                        fontsize=10)
    axes[idx].set_xlabel("Output Value")
    if idx == 0:
        axes[idx].set_ylabel("Count")
    axes[idx].grid(alpha=0.3)

plt.suptitle("Output Value Distributions by Group Count", fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

## 20. Scaling Analysis: Groups vs Model Dimension

How do parameter savings scale as the model dimension increases?

In [ ]:
d_values = [64, 128, 256, 512, 1024, 2048, 4096]
fixed_q_heads = 32  # Fix Q heads

fig, ax = plt.subplots(figsize=(9, 5))

for kv_h, color, ls in [(32, "#2196F3", "-"), (8, "#4CAF50", "--"),
                         (4, "#FF9800", "-."), (1, "#F44336", ":")]:
    param_counts = []
    for dv in d_values:
        if dv >= fixed_q_heads:  # head_dim must be >= 1
            p = count_params_production(dv, fixed_q_heads, kv_h)
            param_counts.append(p)
        else:
            param_counts.append(None)
    
    valid_d = [dv for dv, p in zip(d_values, param_counts) if p is not None]
    valid_p = [p / 1e6 for p in param_counts if p is not None]
    label = {32: "MHA (32 KV)", 1: "MQA (1 KV)"}.get(kv_h, f"GQA ({kv_h} KV)")
    ax.plot(valid_d, valid_p, ls, color=color, linewidth=2, marker="o", label=label)

ax.set_xlabel("Model Dimension")
ax.set_ylabel("Parameters (Millions)")
ax.set_title(f"Attention Parameters vs Model Dimension ({fixed_q_heads} Q heads)")
ax.legend()
ax.grid(alpha=0.3)
ax.set_xscale("log", base=2)
ax.set_yscale("log")
plt.tight_layout()
plt.show()

## 21. Effective Rank of Attention Output

We can measure the "expressiveness" of different configurations by looking at
the effective rank of the output matrix. Higher rank means more diverse
token representations.

In [ ]:
def effective_rank(matrix):
    """Compute effective rank via Shannon entropy of singular values."""
    s = torch.linalg.svdvals(matrix.float())
    s = s / s.sum()  # normalize to distribution
    s = s[s > 1e-10]  # remove zeros
    entropy = -(s * s.log()).sum()
    return torch.exp(entropy).item()

print(f"{'Config':<12} {'Eff. Rank':<14} {'Max Rank':<12} {'Ratio'}")
print("-" * 50)

ranks = []
for g in group_values:
    torch.manual_seed(42)
    model = GQA(d, h_groups=g)
    out = model(x_test).detach()
    # Take batch 0, output is (T, d)
    er = effective_rank(out[0])
    max_rank = min(seq_len, d)
    ranks.append(er)
    label = {1: "MQA", 8: "MHA"}.get(g, f"GQA-{g}")
    print(f"{label:<12} {er:<14.2f} {max_rank:<12} {er/max_rank:.2%}")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
bar_labels = [f"G={g}\n({'MQA' if g==1 else 'MHA' if g==8 else f'GQA-{g}'})" for g in group_values]
ax.bar(bar_labels, ranks, color=colors, edgecolor="black", linewidth=0.5)
ax.axhline(y=min(seq_len, d), color="gray", linestyle="--", alpha=0.5, label="Max rank")
ax.set_ylabel("Effective Rank")
ax.set_title("Effective Rank of Attention Output")
ax.legend()
ax.grid(axis="y", alpha=0.3)
for i, r in enumerate(ranks):
    ax.text(i, r + 0.1, f"{r:.2f}", ha="center", fontsize=9, fontweight="bold")
plt.tight_layout()
plt.show()

## 22. Gradient Flow Comparison

Does the number of groups affect gradient magnitude? This matters for training stability.

In [ ]:
print(f"{'Config':<12} {'Wq Grad Norm':<16} {'Wk Grad Norm':<16} {'Wv Grad Norm'}")
print("-" * 60)

grad_norms = {}
for g in group_values:
    torch.manual_seed(42)
    model = GQA(d, h_groups=g)
    out = model(x_test)
    loss = out.sum()
    loss.backward()
    
    wq_grad = model.Wq.weight.grad.norm().item()
    wk_grad = model.Wk.weight.grad.norm().item()
    wv_grad = model.Wv.weight.grad.norm().item()
    grad_norms[g] = (wq_grad, wk_grad, wv_grad)
    
    label = {1: "MQA", 8: "MHA"}.get(g, f"GQA-{g}")
    print(f"{label:<12} {wq_grad:<16.4f} {wk_grad:<16.4f} {wv_grad:.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))

x_pos = np.arange(len(group_values))
width = 0.25

for i, (proj_name, proj_color) in enumerate([("Wq", "#2196F3"), ("Wk", "#4CAF50"), ("Wv", "#FF9800")]):
    vals = [grad_norms[g][i] for g in group_values]
    ax.bar(x_pos + i * width, vals, width, label=proj_name, color=proj_color,
           edgecolor="black", linewidth=0.3)

ax.set_xticks(x_pos + width)
ax.set_xticklabels([f"G={g}" for g in group_values])
ax.set_ylabel("Gradient Norm")
ax.set_title("Gradient Norms by Projection and Group Count")
ax.legend()
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

## 23. Trade-off Summary Table

Putting it all together: a comprehensive comparison of all configurations.

In [ ]:
# Build a comprehensive summary
print(f"{'Config':<10} {'Groups':<8} {'Params':<10} {'KV Cache':<12} "
      f"{'Cos Sim':<10} {'Eff Rank':<10} {'Grad Norm'}")
print("=" * 75)

for idx, g in enumerate(group_values):
    label = {1: "MQA", 8: "MHA"}.get(g, f"GQA-{g}")
    
    # Params (production, d=64, 8 Q heads)
    params = count_params_production(d, 8, g)
    
    # KV cache ratio
    kv_ratio = g / 8
    
    # Cosine sim to MHA
    cos = cos_sims[idx]
    
    # Effective rank
    rank = ranks[idx]
    
    # Total gradient norm
    total_grad = sum(grad_norms[g])
    
    print(f"{label:<10} {g:<8} {params:<10} {kv_ratio:<12.0%} "
          f"{cos:<10.4f} {rank:<10.2f} {total_grad:.4f}")

## 24. The GQA Efficiency Frontier

Plotting the quality-efficiency trade-off: cosine similarity to MHA
vs memory savings.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

memory_savings = [(1 - g / 8) * 100 for g in group_values]

for idx, g in enumerate(group_values):
    label = {1: "MQA", 8: "MHA"}.get(g, f"GQA-{g}")
    ax.scatter(memory_savings[idx], cos_sims[idx],
              s=200, color=colors[idx], edgecolors="black", linewidth=1, zorder=5)
    ax.annotate(f"{label}\n(G={g})",
               (memory_savings[idx], cos_sims[idx]),
               textcoords="offset points", xytext=(15, 10),
               fontsize=10, fontweight="bold")

ax.plot(memory_savings, cos_sims, "--", color="gray", alpha=0.5, zorder=1)
ax.set_xlabel("KV Cache Memory Savings (%)")
ax.set_ylabel("Cosine Similarity to MHA")
ax.set_title("Quality vs Efficiency Trade-off")
ax.grid(alpha=0.3)

# Highlight sweet spot region
ax.axvspan(50, 80, alpha=0.1, color="green", label="Sweet spot (GQA)")
ax.legend(loc="lower left")
plt.tight_layout()
plt.show()

## 25. Key Insights

### GQA is the sweet spot between MHA quality and MQA efficiency.

**What we learned:**

1. **The GQA spectrum**: MHA and MQA are special cases of GQA. Setting
   `num_kv_heads = num_q_heads` gives MHA; setting `num_kv_heads = 1` gives MQA.
   Anything in between is GQA.

2. **Parameter savings**: GQA reduces K and V projection parameters proportionally
   to the number of KV head groups. With 8 Q heads and 2 KV groups, KV projections
   are 4x smaller.

3. **KV cache savings**: This is the primary motivation. During inference, GQA
   reduces KV cache memory by `1 - (num_kv_heads / num_q_heads)`. For Llama 2 70B,
   this is an 87.5% reduction.

4. **Quality preservation**: With the same weights, GQA produces outputs that
   differ from MHA (since the grouping changes the attention computation),
   but after training, GQA achieves quality close to MHA.

5. **Practical adoption**: Llama 2 70B (8 KV heads for 64 Q heads),
   Mistral 7B (8 KV heads for 32 Q heads), and many other production models
   have validated GQA as a practical design choice.

6. **The key insight**: Most of the representational power in attention comes from
   the Query projections. Keys and Values can be shared across groups of Query heads
   with minimal quality loss, because the diverse Q heads still create different
   attention patterns even when attending to the same K/V.